# AAI-540 Group Project
## Machine Learning-Based RF Intrusion Detection System

### Group 6

This project develops a supervised multi-class machine learning system for classifying radiofrequency (RF) activity using spectrogram images.

The objective is to identify different RF activity categories and provide a repeatable machine learning workflow that supports data preparation, model training, evaluation, deployment, and monitoring.


## 1. Environment Setup

This section imports the libraries required for data inspection, visualization, preprocessing, and machine learning. Random seeds are also defined to support reproducibility across notebook runs.


In [ ]:
# ============================================
# Environment Setup
# ============================================

import os
import random
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.metrics import (
    classification_report,
    confusion_matrix
)

import warnings
warnings.filterwarnings("ignore")

# Reproducibility
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("Environment setup complete.")


## 2. Dataset Location and Access

The Unified Multi-Class RF Intrusion Dataset is stored in Amazon S3 and copied into the SageMaker environment for analysis and model development.

The dataset is organized into predefined training, validation, and test directories, which are used throughout the project.


In [ ]:
# ============================================
# Load Dataset from SageMaker Local Storage
# ============================================

dataset_dir = "/home/sagemaker-user/aai-540-final-project-group6/data/unified_dataset"

train_dir = os.path.join(dataset_dir, "train")
val_dir = os.path.join(dataset_dir, "val")
test_dir = os.path.join(dataset_dir, "test")

print("Dataset directory:", dataset_dir)
print("Training directory:", train_dir)
print("Validation directory:", val_dir)
print("Testing directory:", test_dir)

print("\nDataset contents:")
print(os.listdir(dataset_dir))


## 3. Dataset Structure

This section verifies the dataset directory structure and confirms that the expected training, validation, and test partitions are available before further processing.


In [ ]:
# ============================================
# Inspect Dataset Structure
# ============================================

print("Top-level dataset contents:\n")

for item in sorted(os.listdir(dataset_dir)):
    item_path = os.path.join(dataset_dir, item)

    if os.path.isdir(item_path):
        print(f"[DIR]  {item}")
    else:
        print(f"[FILE] {item}")


In [ ]:
# ============================================
# Define Dataset Splits
# ============================================

train_dir = os.path.join(dataset_dir, "train")
val_dir = os.path.join(dataset_dir, "val")
test_dir = os.path.join(dataset_dir, "test")

print("Training directory:", train_dir)
print("Validation directory:", val_dir)
print("Testing directory:", test_dir)


## 4. RF Activity Classes

The dataset contains six RF activity classes:

- Benign
- Broadband Jam
- Human
- Machine
- Narrowband Jam
- Wildlife

These folder names serve as the class labels for the supervised multi-class classification problem.


In [ ]:
# ============================================
# Identify RF Classes
# ============================================

class_names = sorted([
    folder
    for folder in os.listdir(train_dir)
    if os.path.isdir(os.path.join(train_dir, folder))
])

print(f"Number of classes: {len(class_names)}\n")

for i, class_name in enumerate(class_names):
    print(f"{i}: {class_name}")


## 5. Dataset Distribution

The number of images in each class and dataset split is examined to identify potential class imbalance and confirm the total data volume available for training, validation, and testing.


In [ ]:
# ============================================
# Count Images by Split and Class
# ============================================

def count_images(directory):
    counts = {}

    for class_name in class_names:
        class_dir = os.path.join(directory, class_name)

        counts[class_name] = len([
            file
            for file in os.listdir(class_dir)
            if file.lower().endswith(".png")
        ])

    return counts


train_counts = count_images(train_dir)
val_counts = count_images(val_dir)
test_counts = count_images(test_dir)

print("Class Distribution\n")

print(f"{'Class':<20} {'Train':>8} {'Val':>8} {'Test':>8} {'Total':>8}")
print("-" * 58)

for class_name in class_names:
    train_count = train_counts[class_name]
    val_count = val_counts[class_name]
    test_count = test_counts[class_name]
    total = train_count + val_count + test_count

    print(
        f"{class_name:<20}"
        f"{train_count:>8}"
        f"{val_count:>8}"
        f"{test_count:>8}"
        f"{total:>8}"
    )


In [ ]:
# ============================================
# Dataset Summary
# ============================================

total_train = sum(train_counts.values())
total_val = sum(val_counts.values())
total_test = sum(test_counts.values())

total_images = total_train + total_val + total_test

print("Dataset Summary")
print("-----------------")
print(f"Training images:   {total_train:,}")
print(f"Validation images: {total_val:,}")
print(f"Testing images:    {total_test:,}")
print(f"Total images:      {total_images:,}")


## 6. Class Balance Analysis

Class distribution is visualized to determine whether any RF activity category is overrepresented or underrepresented.

A balanced dataset reduces the likelihood that the model will favor one class simply because it appears more frequently during training.


In [ ]:
# ============================================
# Visualize Class Distribution
# ============================================

totals_by_class = [
    train_counts[c] + val_counts[c] + test_counts[c]
    for c in class_names
]

plt.figure(figsize=(10, 5))
plt.bar(class_names, totals_by_class)

plt.title("RF Spectrogram Class Distribution")
plt.xlabel("RF Activity Class")
plt.ylabel("Number of Images")

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 7. RF Spectrogram Visualization

Sample spectrograms from each RF activity class are displayed to provide a visual understanding of the input data.

These images represent time-frequency characteristics of RF activity that the classification model will learn to distinguish.


In [ ]:
# ============================================
# Display One Spectrogram from Each Class
# ============================================

plt.figure(figsize=(15, 10))

for i, class_name in enumerate(class_names):

    class_dir = os.path.join(train_dir, class_name)

    image_files = [
        file
        for file in os.listdir(class_dir)
        if file.lower().endswith(".png")
    ]

    sample_file = random.choice(image_files)
    image_path = os.path.join(class_dir, sample_file)

    with Image.open(image_path) as image:
        plt.subplot(2, 3, i + 1)
        plt.imshow(image, cmap="gray")
        plt.title(class_name.replace("_", " "))
        plt.axis("off")

plt.suptitle("Sample RF Spectrograms", fontsize=16)
plt.tight_layout()
plt.show()


## 8. Image Properties and Data Quality

The image dimensions and color modes are inspected to verify the consistency of the RF spectrogram data before model development.


In [ ]:
# ============================================
# Inspect Spectrogram Image Properties
# ============================================

sample_class = class_names[0]

sample_files = [
    file for file in os.listdir(os.path.join(train_dir, sample_class))
    if file.lower().endswith(".png")
]

sample_path = os.path.join(
    train_dir,
    sample_class,
    sample_files[0]
)

with Image.open(sample_path) as sample_image:
    print("Class:", sample_class)
    print("Image size:", sample_image.size)
    print("Image mode:", sample_image.mode)
    print("Image format:", sample_image.format)


## 9. Image Consistency Check

A sample of training images is reviewed across all six RF classes to confirm that image dimensions and formats are consistent across the dataset.


In [ ]:
# ============================================
# Check Image Dimensions Across All Classes
# ============================================

image_sizes = Counter()
image_modes = Counter()

samples_per_class = 100
sampled = 0

for class_name in class_names:

    class_dir = os.path.join(train_dir, class_name)

    image_files = [
        file for file in os.listdir(class_dir)
        if file.lower().endswith(".png")
    ]

    sample_files = image_files[:samples_per_class]

    for file in sample_files:

        image_path = os.path.join(class_dir, file)

        with Image.open(image_path) as img:
            image_sizes[img.size] += 1
            image_modes[img.mode] += 1

        sampled += 1

print("Sampled images:", sampled)

print("\nImage dimensions:")
for size, count in image_sizes.items():
    print(size, ":", count)

print("\nImage modes:")
for mode, count in image_modes.items():
    print(mode, ":", count)


## 10. TensorFlow Data Pipeline

The predefined training, validation, and test directories are converted into TensorFlow datasets.

The original spectrogram files remain unchanged at 224 × 224 pixels. For CPU-efficient baseline training in the AWS Academy SageMaker environment, TensorFlow resizes the model inputs to 128 × 128 pixels, loads them as RGB tensors, and processes them in batches of 32.


In [ ]:
# ============================================
# Create TensorFlow Image Datasets
# ============================================

IMG_HEIGHT = 128
IMG_WIDTH = 128
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    color_mode="rgb",
    shuffle=True,
    seed=SEED
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    val_dir,
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    color_mode="rgb",
    shuffle=False
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    color_mode="rgb",
    shuffle=False
)

print("\nClasses:")
print(train_ds.class_names)


## 11. Input Pipeline Optimization

TensorFlow prefetching is used to improve data-loading efficiency by preparing future batches while the model processes the current batch.

This helps reduce input bottlenecks during training.


In [ ]:
# ============================================
# Optimize Input Pipeline
# ============================================

AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.prefetch(buffer_size=AUTOTUNE)

print("TensorFlow datasets ready.")


## 12. Data Preparation Summary

The RF spectrogram dataset has been prepared for model development. The training, validation, and test splits are verified, all six RF activity classes are identified, class balance is evaluated, and image dimensions and formats are checked.

The TensorFlow input pipeline resizes model inputs to 128 × 128 pixels for CPU-efficient baseline training, loads the images as RGB tensors, applies categorical labels for multi-class classification, and uses prefetching to improve data-loading efficiency.

The dataset is now ready for baseline convolutional neural network (CNN) development.


## 13. Baseline CNN Model

A baseline convolutional neural network (CNN) is developed to classify the six RF spectrogram categories.

The baseline model provides an initial performance benchmark that can later be compared against more advanced architectures or transfer-learning approaches.


In [ ]:
# ============================================
# Build Baseline CNN Model
# ============================================

num_classes = len(class_names)

model = tf.keras.Sequential([
    tf.keras.layers.Input(
        shape=(IMG_HEIGHT, IMG_WIDTH, 3)
    ),

    tf.keras.layers.Rescaling(1./255),

    tf.keras.layers.Conv2D(
        32,
        kernel_size=3,
        activation="relu",
        padding="same"
    ),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(
        64,
        kernel_size=3,
        activation="relu",
        padding="same"
    ),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(
        128,
        kernel_size=3,
        activation="relu",
        padding="same"
    ),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.GlobalAveragePooling2D(),

    tf.keras.layers.Dense(
        128,
        activation="relu"
    ),

    tf.keras.layers.Dropout(0.3),

    tf.keras.layers.Dense(
        num_classes,
        activation="softmax"
    )
])

model.summary()


## 14. Model Compilation

The baseline CNN is compiled using the Adam optimizer and categorical cross-entropy loss, which is appropriate for this six-class classification problem. Accuracy is tracked during training as an initial performance metric.


In [ ]:
# ============================================
# Compile Baseline CNN Model
# ============================================

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

print("Baseline CNN compiled successfully.")


## 15. Baseline CNN Training

The baseline CNN is trained using the prepared training dataset and evaluated against the validation dataset after each epoch.

Early stopping is used to reduce unnecessary training if validation loss stops improving, while the best-performing model weights are restored automatically.


## 26. Monitoring Baseline Artifact

The model monitoring baseline is saved as a versioned JSON artifact and stored alongside the trained model in Amazon S3.

This baseline provides reference values for future monitoring of prediction confidence, class distribution, model performance, and potential drift.

In [ ]:
# ============================================
# Save and Upload Monitoring Baseline
# ============================================

monitoring_filename = "monitoring_baseline.json"

# Save monitoring baseline locally
with open(monitoring_filename, "w") as f:
    json.dump(
        monitoring_baseline,
        f,
        indent=4
    )

monitoring_key = (
    f"models/baseline/{model_version}/"
    f"{monitoring_filename}"
)

# Upload monitoring baseline to S3
s3.upload_file(
    monitoring_filename,
    bucket,
    monitoring_key
)

monitoring_s3_uri = (
    f"s3://{bucket}/{monitoring_key}"
)

print("Monitoring baseline saved locally:")
print(monitoring_filename)

print("\nMonitoring baseline uploaded:")
print(monitoring_s3_uri)